<a href="https://colab.research.google.com/github/ochilovu2010/IOAI/blob/main/IOAI_2026/Ghost_Of_Machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2026/blob/main/Individual-Contest/5_Ghost_of_Machine/code/baseline/solution.ipynb)

# Ghost of the Machine — baseline on Google Colab

Run all (GPU runtime recommended). The first cell fetches the [dataset](https://huggingface.co/datasets/IOAI-official/ioai-2026-ghost-of-the-machine) and recreates the contest file layout; every cell after it is the original, untouched baseline.

> **Unofficial educational version** — provided so the task can be used outside the contest environment, reading the data directly from the Hugging Face dataset. The official contest artifacts are preserved in `code/baseline-original/` and `code/grading-original/`.

In [ ]:
# ============================ Colab setup (added) ============================
# Downloads the task dataset from Hugging Face and lays it out exactly as the
# contest environment did. Everything below this cell is the original baseline.
import os, sys, shutil, subprocess
from pathlib import Path
def sh(c): print('+',c); subprocess.run(c, shell=True, check=True)
sh('pip -q install huggingface_hub')

from huggingface_hub import snapshot_download
DATA = Path(snapshot_download("IOAI-official/ioai-2026-ghost-of-the-machine", repo_type="dataset"))
def link(src, dst):
    dst = Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink() or dst.exists(): return
    os.symlink(src, dst)
# merged dataset root: split folders + *_answers files, public and private together
DSROOT = Path("/content/dsroot").resolve()
for sub in ("public","private"):
    d = DATA/sub
    if d.is_dir():
        for child in d.iterdir():
            link(child, DSROOT/child.name)
print("dataset root:", DSROOT, "->", sorted(p.name for p in DSROOT.iterdir()))

EVAL_SPLIT = "test_leaderboard_a"
ws = Path.cwd()
# layout exactly as the contest mounted it: train answers INSIDE dataset/train/
link(DSROOT/"train"/"data.jsonl", ws/"dataset"/"train"/"data.jsonl")
link(DSROOT/"train_answers.jsonl", ws/"dataset"/"train"/"answers.jsonl")
link(DSROOT/EVAL_SPLIT/"data.jsonl", ws/"dataset"/"test_public"/"data.jsonl")
print("ready: dataset/ (test_public ->", EVAL_SPLIT + ")")


+ pip -q install huggingface_hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

dataset root: /content/dsroot -> ['pretest', 'pretest_answers.jsonl', 'pretrain', 'pretrain_answers.jsonl', 'test_leaderboard_a', 'test_leaderboard_a_answers.jsonl', 'test_leaderboard_b', 'test_leaderboard_b_answers.jsonl', 'train', 'train_answers.jsonl']
ready: dataset/ (test_public -> test_leaderboard_a)


# Ghost of the Machine — baseline

A trivial reference baseline: estimate one number from `dataset/train/` — the average
position of the boundary as a fraction of the passage length — and predict that
same fraction of the length for every test passage.

It exists as a runnable template for the contract: read `dataset/test_public/data.jsonl`,
write `answers.jsonl` at the repository root. Replace the logic below with
your own method.


In [ ]:
import json, os

TRAIN_DIR = "dataset/train"
TEST_DIR  = "dataset/test_public"
OUTPUT    = "answers.jsonl"

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

# ---- "train": average boundary position as a fraction of passage length ----
train_rows = read_jsonl(f"{TRAIN_DIR}/data.jsonl")
train_ans = {r["id"]: r["boundary_char_index"] for r in read_jsonl(f"{TRAIN_DIR}/answers.jsonl")}
fracs = [train_ans[r["id"]] / len(r["text"]) for r in train_rows if len(r["text"]) > 0]
mean_frac = sum(fracs) / len(fracs)
print(f"mean boundary fraction from {len(fracs)} train passages: {mean_frac:.4f}")

# ---- predict: same fraction for every test passage ----
test_rows = read_jsonl(f"/content/dsroot/test_leaderboard_b/data.jsonl")
preds = {r["id"]: int(mean_frac * len(r["text"])) for r in test_rows}



mean boundary fraction from 1221 train passages: 0.5952


In [ ]:
train_text = [x['text'] for x in train_rows]
train_answers = [answers for x, answers in train_ans.items()]
len(train_text), len(train_answers)

(1221, 1221)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score

In [ ]:
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = CountVectorizer()
model = LogisticRegression()

In [ ]:
data = []
labels = []
for text, answer in zip(train_text, train_answers):
  human = text[:answer].split('.')
  ai = text[answer:].split('.')
  data.extend(human)
  labels.extend(len(human)*[1])
  data.extend(ai)
  labels.extend(len(ai)*[0])

In [ ]:
  print(len(data), len(labels))
  print(data[0], labels[0])

38380 38380
Tello bequeathed his extensive landed and movable wealth, which was concentrated in the lower Surselva between Flims and Trun, to the Abbey 1


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(data, labels)
vectorized_data = vectorizer.fit_transform(x_train)
model.fit(vectorized_data, y_train)
local_prediction = model.predict(vectorizer.transform(x_test))
score = accuracy_score(local_prediction, y_test)
print(score)

0.9350703491401772


In [ ]:

test_text = [x['text'] for x in test_rows]

vectorized_test = vectorizer.transform(test_text)

In [ ]:
mypredictions = []
for x in test_text:
    testlist = x.split('.')
    predictions = model.predict(vectorizer.transform(testlist))
    char_pos = None
    accumulated_len = 0
    for idx, (sentence, pred) in enumerate(zip(testlist, predictions)):
        if pred == 0 and predictions[idx+1] == 0:
            char_pos = accumulated_len
            break
        accumulated_len += len(sentence) + 1

    mypredictions.append(char_pos+1)

In [ ]:

len(mypredictions)

380

In [ ]:
with open(OUTPUT, "w", encoding="utf-8") as f:
    for r, answer in zip(test_rows, mypredictions):
        f.write(json.dumps({"id": r["id"], "boundary_char_index": answer}) + "\n")
print(f"wrote {OUTPUT}: {len(mypredictions)} predictions")

# self-score when the dev answers are present (absent in the hidden grading set)
ans_path = '/content/dsroot/test_leaderboard_b_answers.jsonl'
if os.path.exists(ans_path):
    import math
    pred_dict = {r["id"]: answer for r, answer in zip(test_rows, mypredictions)}
    gt = {r["id"]: r["boundary_char_index"] for r in read_jsonl(ans_path)}
    scores = [
        math.exp(-abs(pred_dict[i] - gt[i]) / 100.0)
        for i in gt
        if i in pred_dict and pred_dict[i] is not None
    ]
    if scores:
        print(f"self-score: {sum(scores)/len(scores):.4f}")
    else:
        print("No valid predictions to score.")

wrote answers.jsonl: 380 predictions
self-score: 0.8407
